# Entrenament i experiment diferents models

## Entrenament models (nano, small i medium)

In [ ]:
from ultralytics import YOLO
import time
import pandas as pd
import os

# Definim les rutes
ruta_yaml = '/content/drive/MyDrive/Tfg/Dataset/dataset_gt/data.yaml'
ruta_projecte_drive = '/content/drive/MyDrive/Tfg/Dataset/dataset_gt/comparativa_models'

os.makedirs(ruta_projecte_drive, exist_ok=True)

# Models que volem entrenar
models_a_entrenar = ['yolov8n-seg.pt', 'yolov8s-seg.pt', 'yolov8m-seg.pt']
registre_temps = []

for nom_model in models_a_entrenar:
    print(f"\nEntrenant el model: {nom_model}")

    model = YOLO(nom_model)
    nom_carpeta = nom_model.replace('.pt', '')

    # Comencem a comptar el temps
    t_inici = time.time()

    model.train(
        data=ruta_yaml,
        epochs=50,
        imgsz=640,
        device=0,
        plots=True,
        project=ruta_projecte_drive,
        name=nom_carpeta,
        exist_ok=True
    )

    # Parem de comptar el temps
    t_fi = time.time()
    temps_minuts = (t_fi - t_inici) / 60

    # Guardem quant ha trigat
    registre_temps.append({
        'Arquitectura': nom_carpeta.upper(),
        'Temps Entrenament [min]': round(temps_minuts, 2)
    })
    print(f"Temps per {nom_carpeta}: {temps_minuts:.2f} minuts")

# Guardem els resultats en un arxiu CSV
ruta_temps_csv = os.path.join(ruta_projecte_drive, 'temps_entrenament.csv')
pd.DataFrame(registre_temps).to_csv(ruta_temps_csv, index=False)

print("\nEntrenaments acabats i dades guardades.")

## Càlcul mètriques models

In [ ]:
import pandas as pd
import os

# Configurem les rutes
ruta_base = '/content/drive/MyDrive/Tfg/Dataset/dataset_gt/comparativa_models'
models_entrenats = ['yolov8n-seg', 'yolov8s-seg', 'yolov8m-seg']
dades_comparativa = []

# Llegim el CSV amb els temps d'entrenament
ruta_temps = os.path.join(ruta_base, 'temps_entrenament.csv')
df_temps = pd.read_csv(ruta_temps) if os.path.exists(ruta_temps) else pd.DataFrame()

# Traiem les dades de cada model
for model in models_entrenats:
    ruta_csv = os.path.join(ruta_base, model, 'results.csv')

    if os.path.exists(ruta_csv):
        df = pd.read_csv(ruta_csv)
        df.columns = df.columns.str.strip()

        millor_map50_mascara = df['metrics/mAP50(M)'].max() * 100
        millor_map50_95_mascara = df['metrics/mAP50-95(M)'].max() * 100

        # Busquem quant ha trigat aquest model en concret
        temps_model = "N/D"
        if not df_temps.empty:
            fila_temps = df_temps[df_temps['Arquitectura'] == model.upper()]
            if not fila_temps.empty:
                temps_model = fila_temps.iloc[0]['Temps Entrenament [min]']

        dades_comparativa.append({
            'Arquitectura': model.upper(),
            'Temps Entrenament [min]': temps_model,
            'mAP50 (Màscara) [%]': round(millor_map50_mascara, 2),
            'mAP50-95 (Màscara) [%]': round(millor_map50_95_mascara, 2)
        })

# Mostrem la taula amb els resultats
if len(dades_comparativa) > 0:
    df_resultats = pd.DataFrame(dades_comparativa)

    print("\nResultats dels models:")
    print(df_resultats.to_string(index=False))

## Test diferents paràmetres (models nano i small)

In [ ]:
import os
import shutil
import time
import pandas as pd
import cv2
import yaml
from ultralytics import YOLO
from ultralytics.utils import ROOT

# Rutes i variables inicials
ruta_base_models = '/content/drive/MyDrive/Tfg/Dataset/dataset_gt/comparativa_models'
ruta_video_entrada = '/content/drive/MyDrive/Tfg/Dataset/dataset_gt/Video_test_1/video_retallat_1_5_min.mp4'
ruta_sortida_drive = '/content/drive/MyDrive/Tfg/Dataset/dataset_gt/results_seguimiento/experiment_hiperparametres'

os.makedirs(ruta_sortida_drive, exist_ok=True)
arquitectures = ['yolov8n-seg', 'yolov8s-seg']
trackers_base = ['bytetrack.yaml', 'botsort.yaml']

# Valors que volem provar
valors_memoria = [30, 90, 120]
valors_confianca = [0.1, 0.3, 0.5, 0.7, 0.9]

dades_experiments = []

print("\nComencem la prova de paràmetres (memòria i confiança)")

# Creem arxius de configuració per als trackers
print("Creant els arxius de configuració...")
trackers_personalitzats = []

for tracker in trackers_base:
    ruta_tracker_original = os.path.join(ROOT, 'cfg', 'trackers', tracker)
    with open(ruta_tracker_original, 'r') as f:
        config_tracker = yaml.safe_load(f)

    nom_base = tracker.replace('.yaml', '')

    for memoria in valors_memoria:
        config_tracker['track_buffer'] = memoria
        nou_nom = f"custom_{nom_base}_buf{memoria}.yaml"
        ruta_nou_yaml = os.path.join('/content', nou_nom)

        with open(ruta_nou_yaml, 'w') as f:
            yaml.dump(config_tracker, f)

        trackers_personalitzats.append({
            'arxiu': ruta_nou_yaml,
            'base': nom_base.upper(),
            'memoria': memoria
        })

captura = cv2.VideoCapture(ruta_video_entrada)
total_fotogrames = int(captura.get(cv2.CAP_PROP_FRAME_COUNT))
captura.release()

# Provem totes les combinacions
for arquitectura in arquitectures:
    ruta_pesos = os.path.join(ruta_base_models, arquitectura, 'weights', 'best.pt')
    if not os.path.exists(ruta_pesos): continue

    model = YOLO(ruta_pesos)

    for cfg in trackers_personalitzats:
        tracker_yaml = cfg['arxiu']
        tracker_base = cfg['base']
        memoria_frames = cfg['memoria']

        # Bucle per provar la confiança
        for conf_val in valors_confianca:
            nom_experiment = f"{arquitectura}_{tracker_base}_BUF{memoria_frames}_CONF{conf_val}"
            ruta_video_final = os.path.join(ruta_sortida_drive, f"Resultat_{nom_experiment}.mp4")

            print(f"Provant {arquitectura} amb {tracker_base} (Buffer: {memoria_frames}, Confiança: {conf_val})... ", end="")
            t_inici = time.time()

            ids_unics = set()

            resultats = model.track(
                source=ruta_video_entrada,
                conf=conf_val,         # Posem la confiança
                tracker=tracker_yaml,
                save=True,
                project='/content/runs/tracking_output',
                name=nom_experiment,
                exist_ok=True,
                stream=True,
                verbose=False
            )

            for r in resultats:
                if r.boxes is not None and r.boxes.id is not None:
                    ids_actuals = r.boxes.id.cpu().numpy().astype(int)
                    ids_unics.update(ids_actuals)

            t_fi = time.time()
            temps_total = t_fi - t_inici
            fps_processament = total_fotogrames / temps_total if temps_total > 0 else 0
            total_ids_generats = len(ids_unics)

            print(f"Fet! s'han generat {total_ids_generats} IDs.")

            dades_experiments.append({
                'Arquitectura': arquitectura.upper(),
                'Tracker': tracker_base,
                'Memòria [Frames]': memoria_frames,
                'Confiança': conf_val,
                'Velocitat [FPS]': round(fps_processament, 2),
                'Total IDs Generats': total_ids_generats
            })

            base_search = f'/content/runs/tracking_output/{nom_experiment}'
            if os.path.exists(base_search):
                for root, dirs, files in os.walk(base_search):
                    for f in files:
                        if f.endswith(('.mp4', '.avi')):
                            fitxer_generat = os.path.join(root, f)
                            if os.path.getsize(fitxer_generat) > 0:
                                shutil.move(fitxer_generat, ruta_video_final)
                                break

# Guardem els resultats
if len(dades_experiments) > 0:
    df_resultats = pd.DataFrame(dades_experiments)
    ruta_csv_final = os.path.join(ruta_sortida_drive, 'informe_hiperparametres.csv')
    df_resultats.to_csv(ruta_csv_final, index=False)

    print("\nResultats de la prova de paràmetres:")
    print(df_resultats.to_string(index=False))